# Trabajo Práctico Final — Predicción de ACV
**Autores:** Mauro Virgilio Blanc · Juan Pablo Imbrogno · Sofía Belén Caselli · Miguel Angel Leiva Martinez · Andrea Viviana Ferenaz

---

## 1. Introducción

El problema abordado corresponde a una **tarea de clasificación supervisada**, cuyo objetivo es predecir la probabilidad de que un paciente sufra un accidente cerebrovascular (ACV) a partir de variables demográficas y clínicas (edad, hipertensión, nivel de glucosa, tipo de trabajo, tabaquismo, entre otras).

La variable objetivo `stroke` es **binaria** (0 = no tuvo ACV, 1 = tuvo ACV) y presenta un fuerte **desbalance de clases** (~5% positivos). Por ello, se priorizaron algoritmos robustos frente a este tipo de distribución y se aplicaron técnicas de *oversampling* (SMOTE) durante el entrenamiento para mejorar la capacidad de detección de la clase minoritaria.

## 2. Importaciones y Configuración Global

In [ ]:
# === Librerías estándar ===
import warnings

# === Análisis y manipulación ===
import numpy as np
import pandas as pd

# === Visualización ===
import matplotlib.pyplot as plt
import seaborn as sns

# === Preprocesamiento ===
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# === Modelos ===
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from xgboost import XGBClassifier

# === Imbalanced learning ===
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# === Evaluación y selección ===
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    auc,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split

# === Configuración global ===
RANDOM_STATE = 42
DATA_PATH = "stroke-data.csv"
TEST_SIZE = 0.20
CV_FOLDS = 5
RECALL_TARGET = 0.90        # umbral mínimo de recall para screening clínico

warnings.filterwarnings("ignore", category=UserWarning)

# Estilo de visualización unificado
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 100})

## 3. Carga y Exploración del Dataset

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape}")
display(df.head())
display(df.dtypes.rename("dtype").to_frame())

In [ ]:
# --- Valores faltantes ---
missing = df.isnull().sum()
missing_pct = df.isnull().mean().mul(100).round(2)
display(
    pd.concat([missing, missing_pct], axis=1, keys=["n_nulos", "% nulos"])
    .query("n_nulos > 0")
)

# --- Duplicados ---
n_dup = df.duplicated().sum()
print(f"Filas duplicadas: {n_dup}")

# --- Cardinalidad ---
print("\nValores únicos por columna:")
display(df.nunique().rename("n_unique").to_frame().T)

In [ ]:
# --- Distribución de la variable objetivo ---
stroke_rate = df["stroke"].mean()
print(f"Prevalencia de ACV: {stroke_rate:.4f} ({stroke_rate*100:.2f}%)")

fig, ax = plt.subplots(figsize=(5, 3))
df["stroke"].value_counts().rename({0: "No ACV", 1: "ACV"}).plot(
    kind="bar", ax=ax, color=["steelblue", "tomato"]
)
ax.set_title("Distribución de la variable objetivo")
ax.set_ylabel("Cantidad")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# --- Análisis de correlación numérica ---
corr = df.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", ax=ax, linewidths=0.5)
ax.set_title("Correlación entre variables numéricas")
plt.tight_layout()
plt.show()

In [ ]:
# --- Distribución de variables categóricas ---
cat_cols = ["gender", "ever_married", "work_type", "Residence_type", "smoking_status"]
for col in cat_cols:
    print(f"\n{col}:")
    display(df[col].value_counts(normalize=True).mul(100).round(2).rename("%").to_frame())

In [ ]:
# --- Varianza de variables numéricas ---
numeric_cols = df.select_dtypes(include=["float64", "int64"])
selector = VarianceThreshold(threshold=0.01)
selector.fit(numeric_cols)

cols_ok = numeric_cols.columns[selector.get_support()].tolist()
cols_baja_varianza = numeric_cols.columns[~selector.get_support()].tolist()

print("Columnas con varianza suficiente:", cols_ok)
print("Columnas eliminadas por baja varianza:", cols_baja_varianza)

## 4. Limpieza y Feature Engineering

In [ ]:
df = df.drop(columns=["id"])  # No tiene valor predictivo

# --- Nuevas variables derivadas ---

# Grupos etarios interpretables
df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 30, 50, 70, 120],
    labels=["Joven", "Adulto", "Mayor", "Anciano"]
).astype(str)

# Recorte de outliers de glucosa
df["avg_glucose_level"] = np.clip(df["avg_glucose_level"], 50, 300)

# Combinación de factores de riesgo cardiovascular
df["has_risk_factors"] = (
    (df["hypertension"] == 1) | (df["heart_disease"] == 1)
).astype(int)

# Codificación binaria del género
# "Other" (<0.02%) se absorbe en la mayoría (Female=0 / Male=1)
df["is_female"] = (df["gender"] == "Female").astype(int)
df = df.drop(columns=["gender"])

# Unificar categorías marginales en work_type
# "Never_worked" (0.4%) comparte perfil de riesgo con "children" (13.4%)
df["work_type"] = df["work_type"].replace({"Never_worked": "children"})

print("Shape final después del feature engineering:", df.shape)
display(df.head(3))

## 5. Split y Preprocesamiento
> **Nota sobre data leakage:** la mediana de BMI y los parámetros de `StandardScaler` / `OneHotEncoder` se calculan **únicamente** sobre `X_train`, luego se aplican a `X_test` sin refitting.

In [ ]:
X = df.drop("stroke", axis=1)
y = df["stroke"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

# Imputación de BMI usando la mediana del train (evita leakage)
bmi_median = X_train["bmi"].median()
X_train = X_train.copy()
X_test = X_test.copy()
X_train["bmi"] = X_train["bmi"].fillna(bmi_median)
X_test["bmi"] = X_test["bmi"].fillna(bmi_median)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Positivos en train: {y_train.mean():.3%} | en test: {y_test.mean():.3%}")

In [ ]:
# === Preprocessors reutilizables ===

CAT_FEATURES = ["smoking_status", "age_group"]
# is_female y has_risk_factors son binarias: no necesitan OneHotEncoder ni escalar
NUM_FEATURES = (
    X.select_dtypes(exclude="object")
    .columns
    .drop("is_female")
    .tolist()
)

def build_preprocessor(num_cols=NUM_FEATURES, cat_cols=CAT_FEATURES):
    """Construye un ColumnTransformer con scaler + OHE."""
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), num_cols),
            ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols),
        ],
        remainder="passthrough",  # mantiene columnas no especificadas (is_female, etc.)
    )


# Preprocessor para Logistic Regression (sin columnas ruidosas)
COLS_RUIDO_LR = ["heart_disease", "bmi", "smoking_status", "age_group"]
NUM_FEATURES_LR = [
    c for c in X.select_dtypes(exclude="object").columns.drop("is_female").tolist()
    if c not in COLS_RUIDO_LR
]
CAT_FEATURES_LR = [c for c in CAT_FEATURES if c not in COLS_RUIDO_LR]
preprocessor_lr = build_preprocessor(NUM_FEATURES_LR, CAT_FEATURES_LR)

# Preprocessor genérico
preprocessor_gen = build_preprocessor()

print("Numéricas (LR):", NUM_FEATURES_LR)
print("Categóricas (LR):", CAT_FEATURES_LR)

## 6. Análisis de Variables Ruidosas

In [ ]:
def get_lr_feature_importance(pipe, num_features, cat_features):
    """Extrae los coeficientes de una Regresión Logística dentro de un Pipeline."""
    ohe = pipe.named_steps["preprocessor"].named_transformers_["cat"]
    ohe_names = ohe.get_feature_names_out(cat_features).tolist()
    all_features = num_features + ohe_names
    coefs = pipe.named_steps["model"].coef_[0]
    return (
        pd.DataFrame({"feature": all_features, "coef": coefs, "abs_coef": np.abs(coefs)})
        .sort_values("abs_coef", ascending=False)
        .reset_index(drop=True)
    )


# Modelo de diagnóstico (L1 fuerza coefs cero en variables ruidosas)
pipe_diag = Pipeline(steps=[
    ("preprocessor", preprocessor_gen),
    ("model", LogisticRegression(
        penalty="l1", solver="liblinear",
        C=0.1, max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])
pipe_diag.fit(X_train, y_train)

importancia_lr = get_lr_feature_importance(pipe_diag, NUM_FEATURES, CAT_FEATURES)
display(importancia_lr)

# Conclusión visible en tabla:
# heart_disease, bmi, smoking_status_formerly → coef≈0 → candidatas a exclusión en LR

In [ ]:
# --- Importancias en XGBoost ---
pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]

pipe_xgb_diag = Pipeline(steps=[
    ("preprocessor", preprocessor_gen),
    ("model", XGBClassifier(
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        scale_pos_weight=pos_weight,
        use_label_encoder=False,
        colsample_bytree=0.7,
        gamma=0.3,
        learning_rate=0.01,
        max_depth=3,
        n_estimators=100,
    )),
])
pipe_xgb_diag.fit(X_train, y_train)

ohe_xgb = pipe_xgb_diag.named_steps["preprocessor"].named_transformers_["cat"]
ohe_cols_xgb = ohe_xgb.get_feature_names_out(CAT_FEATURES).tolist()
all_features_xgb = NUM_FEATURES + ohe_cols_xgb

importancia_xgb = (
    pd.DataFrame({
        "feature": all_features_xgb,
        "importance": pipe_xgb_diag.named_steps["model"].feature_importances_,
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
display(importancia_xgb)

# Features con importancia < 1%
features_ruido_xgb = importancia_xgb.query("importance < 0.01")["feature"].tolist()
print("Features con baja importancia en XGB:", features_ruido_xgb)

## 7. Entrenamiento y Validación Cruzada de Modelos

In [ ]:
def categorizar_riesgo_array(prob_array: np.ndarray) -> np.ndarray:
    """
    Categoriza probabilidades en Bajo / Medio / Alto según percentiles 70 y 90.

    Parameters
    ----------
    prob_array : np.ndarray
        Array de probabilidades de la clase positiva (valores en [0, 1]).

    Returns
    -------
    np.ndarray of str
        Etiqueta de riesgo para cada observación.
    """
    p70 = np.percentile(prob_array, 70)
    p90 = np.percentile(prob_array, 90)
    conditions = [
        prob_array < p70,
        (prob_array >= p70) & (prob_array < p90),
        prob_array >= p90,
    ]
    choices = ["Bajo", "Medio", "Alto"]
    return np.select(conditions, choices)

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)

# Clasificadores base para VotingClassifier (necesitan preprocessor propio)
# NOTA: se usan los modelos con preprocessor_gen dentro del VotingClassifier
knn_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=7)),
])

models_base_for_voting = [
    ("lr", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
    ("rf", RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE)),
    ("xgb", XGBClassifier(
        eval_metric="logloss", scale_pos_weight=pos_weight,
        use_label_encoder=False, colsample_bytree=0.7, gamma=0.3,
        learning_rate=0.01, max_depth=3, n_estimators=100,
        random_state=RANDOM_STATE,
    )),
    ("cat", CatBoostClassifier(verbose=0, class_weights=[1, pos_weight], random_state=RANDOM_STATE)),
    ("svm", SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_STATE)),
    ("knn", knn_pipe),
]

# Catálogo de modelos con su preprocessor asignado
# Formato: (nombre, clasificador, preprocessor)
MODEL_CATALOG = [
    (
        "LogisticRegression",
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
        preprocessor_lr,
    ),
    (
        "RandomForest",
        RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        preprocessor_gen,
    ),
    (
        "XGBoost",
        XGBClassifier(
            eval_metric="logloss", scale_pos_weight=pos_weight,
            use_label_encoder=False, colsample_bytree=0.7, gamma=0.3,
            learning_rate=0.01, max_depth=3, n_estimators=100,
            random_state=RANDOM_STATE,
        ),
        preprocessor_gen,
    ),
    (
        "CatBoost",
        CatBoostClassifier(verbose=0, class_weights=[1, pos_weight], random_state=RANDOM_STATE),
        preprocessor_gen,
    ),
    (
        "SVM",
        SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_STATE),
        preprocessor_gen,
    ),
    (
        "KNN",
        knn_pipe,
        preprocessor_gen,
    ),
    (
        "SoftVoting",
        VotingClassifier(estimators=models_base_for_voting, voting="soft"),
        preprocessor_gen,
    ),
]

results = []
trained_pipes: dict[str, ImbPipeline] = {}
y_pred_proba: dict[str, np.ndarray] = {}
y_riesgo_dict: dict[str, np.ndarray] = {}

print("=== Validación cruzada con SMOTE (scoring=roc_auc) ===\n")

for name, clf, prep in MODEL_CATALOG:
    pipe = ImbPipeline(steps=[
        ("preprocessor", prep),
        ("smote", smote),
        ("model", clf),
    ])

    scores = cross_val_score(pipe, X_train, y_train, cv=CV_FOLDS, scoring="roc_auc", n_jobs=-1)

    print(f"{name:20s} AUC: {scores.mean():.4f} ± {scores.std():.4f}")

    results.append({"Modelo": name, "AUC promedio": scores.mean(), "AUC std": scores.std()})

    # Entrenar sobre el conjunto completo de train para evaluar en test
    pipe.fit(X_train, y_train)
    trained_pipes[name] = pipe

    try:
        y_prob = pipe.predict_proba(X_test)[:, 1]
        y_pred_proba[name] = y_prob
        y_riesgo_dict[name] = categorizar_riesgo_array(y_prob)
    except AttributeError:
        print(f"  ⚠ {name} no soporta predict_proba — se omite en curvas ROC.")

resultados_df = (
    pd.DataFrame(results)
    .sort_values("AUC promedio", ascending=False)
    .reset_index(drop=True)
)

print("\n===== TABLA COMPARATIVA =====")
display(resultados_df)

## 8. Visualizaciones Comparativas

In [ ]:
# --- Curvas ROC ---
fig, ax = plt.subplots(figsize=(8, 6))

for name, y_prob in y_pred_proba.items():
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"{name} (AUC={roc_auc:.3f})")

ax.plot([0, 1], [0, 1], "--", color="gray", label="Aleatorio")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Curvas ROC — todos los modelos")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# --- Barras AUC con diferencia respecto al mejor ---
best_auc = resultados_df["AUC promedio"].max()

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(resultados_df["Modelo"], resultados_df["AUC promedio"])

for i, bar in enumerate(bars):
    auc_val = resultados_df.loc[i, "AUC promedio"]
    is_best = auc_val == best_auc
    bar.set_color("tomato" if is_best else "steelblue")
    label = f"Mejor\n{auc_val:.3f}" if is_best else f"-{best_auc - auc_val:.3f}"
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
        label, ha="center", va="bottom", fontsize=9,
    )

ax.set_ylim(0, best_auc + 0.12)
ax.set_ylabel("AUC promedio (CV)")
ax.set_title("Comparación de modelos")
ax.set_xticklabels(resultados_df["Modelo"], rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 9. Optimización del Modelo Seleccionado (Regresión Logística)

In [ ]:
# GridSearchCV con el mismo preprocessor usado para LR
pipe_lr_gs = Pipeline(steps=[
    ("preprocessor", preprocessor_gen),  # reutilizamos el genérico para la búsqueda
    ("model", LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE
    )),
])

param_grid = {
    "model__penalty": ["l1", "l2"],
    "model__C": [0.1, 1.0, 3.0, 10.0],
    "model__solver": ["liblinear"],  # obligatorio para L1
}

grid_search = GridSearchCV(
    pipe_lr_gs, param_grid,
    cv=CV_FOLDS, scoring="roc_auc",
    n_jobs=-1, verbose=1,
)
grid_search.fit(X_train, y_train)

print(f"Mejores parámetros : {grid_search.best_params_}")
print(f"Mejor AUC (CV)     : {grid_search.best_score_:.4f}")

In [ ]:
# Modelo final: mejores hiperparámetros ya ajustados sobre todo X_train
best_lr = grid_search.best_estimator_
best_lr.fit(X_train, y_train)

y_prob_final = best_lr.predict_proba(X_test)[:, 1]
auc_test = roc_auc_score(y_test, y_prob_final)
print(f"AUC en TEST (modelo final): {auc_test:.4f}")

## 10. Umbral Óptimo para Screening Clínico

In [ ]:
precision_arr, recall_arr, thresholds_arr = precision_recall_curve(y_test, y_prob_final)

# Seleccionar el threshold más alto que garantiza recall >= RECALL_TARGET
mask = recall_arr[:-1] >= RECALL_TARGET
best_threshold = float(thresholds_arr[mask].max()) if mask.any() else 0.5

print(f"Umbral óptimo para recall ≥ {RECALL_TARGET:.0%}: {best_threshold:.5f}")

# Predicciones con el umbral ajustado
y_pred_screening = (y_prob_final >= best_threshold).astype(int)

# Comparación de umbrales
y_pred_default = (y_prob_final >= 0.5).astype(int)

print("\n=== Umbral 0.50 ===")
print(classification_report(y_test, y_pred_default, target_names=["No ACV", "ACV"]))

print(f"=== Umbral {best_threshold:.3f} (optimizado) ===")
print(classification_report(y_test, y_pred_screening, target_names=["No ACV", "ACV"]))

In [ ]:
# Matrices de confusión lado a lado
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, y_pred, title in zip(
    axes,
    [y_pred_default, y_pred_screening],
    ["Umbral estándar (0.50)", f"Umbral optimizado ({best_threshold:.3f})"],
):
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, y_pred),
        display_labels=["No ACV", "ACV"],
    ).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(title)

plt.suptitle("Comparación de matrices de confusión", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 11. Análisis de Riesgo — Distribución y Calibración

In [ ]:
# Percentiles calculados una sola vez (sobre el set de test final)
p70 = np.percentile(y_prob_final, 70)
p90 = np.percentile(y_prob_final, 90)

df_resultados = pd.DataFrame({
    "y_real": y_test.values,
    "probabilidad": y_prob_final,
    "pred_binaria": y_pred_screening,
    "riesgo": categorizar_riesgo_array(y_prob_final),
})

display(df_resultados.head())

In [ ]:
CATEGORIAS = ["Bajo", "Medio", "Alto"]
COLORES = {"Bajo": "#2ca02c", "Medio": "#ff7f0e", "Alto": "#d62728"}

proporciones = df_resultados["riesgo"].value_counts(normalize=True).reindex(CATEGORIAS)
tasas_positivos = [
    df_resultados.loc[df_resultados["riesgo"] == nivel, "y_real"].mean()
    for nivel in CATEGORIAS
]

fig, ax1 = plt.subplots(figsize=(8, 5))

bars = ax1.bar(
    CATEGORIAS, proporciones.values,
    color=[COLORES[c] for c in CATEGORIAS], alpha=0.85, edgecolor="black",
)
ax1.set_ylabel("Proporción de pacientes")
ax1.set_xlabel("Categoría de riesgo")
ax1.set_ylim(0, 1)
ax1.set_title("Distribución de riesgo y tasa real de ACV", fontsize=13, fontweight="bold")

for bar, prop in zip(bars, proporciones.values):
    ax1.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() / 2,
        f"{prop*100:.1f}%", ha="center", va="center",
        color="white", fontweight="bold",
    )

ax2 = ax1.twinx()
ax2.plot(CATEGORIAS, tasas_positivos, "o--", lw=2, color="navy", label="Tasa real ACV")
ax2.set_ylabel("Tasa real de positivos (ACV)")
ax2.set_ylim(0, 1)

for i, t in enumerate(tasas_positivos):
    ax2.text(i, t + 0.03, f"{t*100:.1f}%", ha="center", color="navy",
             fontsize=10, fontweight="bold")

ax2.legend(loc="upper left")
plt.tight_layout()
plt.show()

## 12. Importancia de Variables del Modelo Final

In [ ]:
# Recuperar nombres de columnas transformadas del pipeline final
prep_final = best_lr.named_steps["preprocessor"]

# Numéricas
num_names = prep_final.named_transformers_["num"].get_feature_names_out(NUM_FEATURES).tolist()
# Categóricas
cat_names = prep_final.named_transformers_["cat"].get_feature_names_out(CAT_FEATURES).tolist()
# Passthrough (is_female + has_risk_factors)
passthrough_cols = [
    c for c in X.columns
    if c not in NUM_FEATURES and c not in CAT_FEATURES
]

all_feature_names = num_names + cat_names + passthrough_cols

coefs = best_lr.named_steps["model"].coef_[0]

# Alinear longitudes (el passthrough puede tener distinto orden)
n = min(len(all_feature_names), len(coefs))
importancia_final = (
    pd.DataFrame({
        "feature": all_feature_names[:n],
        "coef": coefs[:n],
        "abs_coef": np.abs(coefs[:n]),
    })
    .sort_values("abs_coef", ascending=True)
)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["tomato" if c > 0 else "steelblue" for c in importancia_final["coef"]]
ax.barh(importancia_final["feature"], importancia_final["coef"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Coeficiente (escala estandarizada)")
ax.set_title("Importancia de variables — Regresión Logística")
plt.tight_layout()
plt.show()

## 13. Conclusión

El modelo final, **Regresión Logística optimizado con SMOTE**, fue elegido como el más adecuado por su excelente balance entre **rendimiento (AUC ≈ 0.83), interpretabilidad y estabilidad**.

A pesar del fuerte desbalance de clases (~5% positivos), el desempeño fue satisfactorio, logrando alta capacidad de detección de la clase positiva (alto *recall*).

**Directrices para implementación práctica:**
1. Usar el modelo de Regresión Logística por su interpretabilidad clínica.
2. Implementar el **ajuste del umbral de decisión** (≈ 0.178) para minimizar Falsos Negativos y priorizar el *screening* de riesgo.
3. Las variables más influyentes (**edad, hipertensión, glucosa, tipo de residencia, tabaquismo, BMI**) coinciden con la evidencia clínica sobre factores de riesgo de ACV.